In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.color import label2rgb
from skimage.measure import label

from interactive_seg_backend.extensions import ExpertSegClassifier
from interactive_seg_backend.file_handling import load_image, load_labels, save_segmentation
from interactive_seg_backend.configs import TrainingConfig, FeatureConfig, ClassInfo
from interactive_seg_backend.utils import PALETTE_RGB_NORM, apply_labels_as_overlay

from interactive_seg_backend import featurise

In [ ]:
img = load_image("../tests/data/3.tif")
labels = load_labels("../tests/data/3_labels.tif")

feature_cfg = FeatureConfig()
_training_cfg = TrainingConfig(feature_config=feature_cfg)
feats = featurise(img, _training_cfg)

In [ ]:
class_infos = [
    ClassInfo(name="solid", value=0, connectivity_target="minimise", desired_volume_fraction=0.4979),
    ClassInfo(name="pore", value=1, connectivity_target=None),
]

In [ ]:
lambdas = [(0, 0), (1, 0), (0, 3), (2, 3)]

preds: list[np.ndarray] = []
for lambd_vf, lambd_conn in lambdas:
    model = ExpertSegClassifier(class_infos=class_infos, n_epochs=100, lambd_conn=lambd_conn, lambd_vf=lambd_vf, extra_args={"max_depth": 6
                                                                                                                             , "eta": 0.1})
    if lambd_vf == 0:
        model.do_vf_loss = False
    if lambd_conn == 0:
        model.do_conn_loss = False

    model.fit(feats, labels)
    pred = model.predict(feats) - 1
    preds.append(pred)

In [ ]:

def hide_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])

def render_title(ax, lambd_vf: float, lambd_conn: float, targ_vf: float, targ_conn: str):
    title = ""
    show_vf = lambd_vf > 0
    show_conn = lambd_conn > 0
    if show_vf:
        title += rf"$VF_{{targ}}={targ_vf:.2f}$, $\lambda_{{VF}}={lambd_vf:.1f}$"
    if show_conn and show_vf:
        title += "\n"
    if show_conn:
        targ_text = "min" if targ_conn == "minimise" else "max"
        title += rf"$Conn_{{targ}}={targ_text}$, $\lambda_{{Conn}}={lambd_conn:.1f}$"
    if not show_vf and not show_conn:
        title = "Base loss"
    ax.set_title(title)

def render_xlabel(ax, pred_vf: float, pred_ccs: int):
    title = ""
    show_vf = True
    show_conn = True
    if show_vf:
        title += rf"$VF_{{pred}}={pred_vf:.2f}$"
    if show_conn and show_vf:
        title += "\n"

    if show_conn:
        title += rf"$N_{{CC}}={pred_ccs}$"
    ax.set_xlabel(title, fontsize=12)
    


N_COLS = 1 + len(preds)

W = 3
fig, axs = plt.subplots(1, N_COLS, figsize=(W * N_COLS, W))

img_with_labels = apply_labels_as_overlay(labels, Image.fromarray(img).convert('RGB'), PALETTE_RGB_NORM[1:])

axs[0].imshow(img_with_labels, interpolation='none')
axs[0].set_title("Image + labels")
axs[0].axis('off')
for ax, pred, (lambd_vf, lambd_conn) in zip(axs[1:], preds, lambdas):
    binary_arr = pred == 0
    pred_vf = np.mean(binary_arr)
    _, n_ccs = label(binary_arr, return_num=True)


    pred_recoloured = label2rgb(pred, colors=PALETTE_RGB_NORM[1:], bg_label=-1)
    ax.imshow(pred_recoloured, interpolation='none')
    render_title(ax, lambd_vf, lambd_conn, targ_vf=class_infos[0].desired_volume_fraction, targ_conn=class_infos[0].connectivity_target)
    render_xlabel(ax, pred_vf=pred_vf, pred_ccs=n_ccs)
    hide_axes(ax)